# 🔑 Credencial de Drive para la actualización automática (se hace una sola vez)

La actualización automática corre en GitHub, no en Colab, así que necesita permiso para leer y escribir en tus carpetas DATA y CONTROL de Drive. Este notebook genera ese permiso (una «credencial» con un *refresh token*). **No la compartas con nadie**: la pegas solo en GitHub → Settings → Secrets.

**Antes (una vez), en https://console.cloud.google.com con tu cuenta de Google:**
1. Crea un proyecto (por ejemplo «pp-automatizacion») y activa la **Google Drive API** (APIs y servicios → Biblioteca).
2. *Pantalla de consentimiento de OAuth*: tipo **Externo**; agrega tu correo como usuario de prueba y luego pulsa **Publicar la app** («En producción»). Si queda «En prueba», la credencial vence a los 7 días.
3. *Credenciales → Crear credenciales → ID de cliente de OAuth* → tipo **App de escritorio**. Copia el ID de cliente y el secreto.
4. En este notebook de Colab, abre **🔑 Secretos** (panel izquierdo) y crea `DRIVE_OAUTH_CLIENT_ID` y `DRIVE_OAUTH_CLIENT_SECRET` con esos valores (activa el acceso del notebook).

**Luego ejecuta la celda**:
- Abre el enlace que aparece y entra con tu cuenta. Google avisará que la app no está verificada: es tuya, pulsa *Configuración avanzada → Ir a …*.
- Al final el navegador mostrará una página que **no carga** (localhost). Copia la dirección completa de esa pestaña y pégala en el cuadro de la celda.
- Se descarga `credencial_drive.json`. Abre ese archivo, copia todo su contenido y pégalo en GitHub: repositorio → **Settings → Secrets and variables → Actions → New repository secret**, con el nombre `GOOGLE_DRIVE_CREDENCIAL`. Después **borra el archivo** de tu computadora y no guardes este notebook con la salida.

In [ ]:
#@title Generar la credencial de Drive { display-mode: "form" }
import json, urllib.parse, requests
from google.colab import userdata, files
cid, csec = userdata.get("DRIVE_OAUTH_CLIENT_ID"), userdata.get("DRIVE_OAUTH_CLIENT_SECRET")
REDIR = "http://localhost"
url = "https://accounts.google.com/o/oauth2/v2/auth?" + urllib.parse.urlencode({
    "client_id": cid, "redirect_uri": REDIR, "response_type": "code", "access_type": "offline", "prompt": "consent",
    "scope": "https://www.googleapis.com/auth/drive"})
print("1) Abre este enlace, entra con tu cuenta y acepta los permisos:\n\n" + url + "\n")
resp = input("2) Pega aquí la dirección completa de la página que no cargó (empieza con http://localhost): ").strip()
code = urllib.parse.parse_qs(urllib.parse.urlparse(resp).query).get("code", [resp])[0]
r = requests.post("https://oauth2.googleapis.com/token", data={"code": code, "client_id": cid, "client_secret": csec,
                                                               "redirect_uri": REDIR, "grant_type": "authorization_code"}, timeout=60).json()
if "refresh_token" not in r:
    raise SystemExit(f"❌ Google no entregó la credencial: {r.get('error')} – {r.get('error_description')}. Vuelve a ejecutar la celda.")
open("/content/credencial_drive.json", "w").write(json.dumps({"type": "authorized_user", "client_id": cid,
                                                               "client_secret": csec, "refresh_token": r["refresh_token"]}))
files.download("/content/credencial_drive.json")
print("✅ Descargado credencial_drive.json. Pega su contenido en GitHub como secreto GOOGLE_DRIVE_CREDENCIAL y borra el archivo.")
